In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Verifiko GPU-në

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


shiko path-in e dataset-it (input files)

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/val.csv
/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/train.csv
/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed/test.csv


instalo evaluate dhe verifiko librarit

In [3]:
!pip install -q evaluate

import transformers
import datasets
print("transformers version:", transformers.__version__)
print("datasets version:", datasets.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 836.2 kB/s eta 0:00:00a 0:00:01
transformers version: 5.0.0
datasets version: 5.0.0


ngarko dataset-in dhe përgatit labels

In [4]:
import pandas as pd

DATA_PATH = '/kaggle/input/datasets/denismustafaraj/kosovo-news-dataset-processed'

train_df = pd.read_csv(f'{DATA_PATH}/train.csv')
val_df = pd.read_csv(f'{DATA_PATH}/val.csv')
test_df = pd.read_csv(f'{DATA_PATH}/test.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

kategorite = sorted(train_df['label'].unique())
label2id = {label: i for i, label in enumerate(kategorite)}
id2label = {i: label for label, i in label2id.items()}

print("\nLabel mapping:")
print(label2id)

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

train_df.head()

Train: (75480, 4)
Validation: (16174, 4)
Test: (16175, 4)

Label mapping:
{'Botë': 0, 'Ekonomi': 1, 'Kulturë': 2, 'Kuriozitete': 3, 'Lifestyle': 4, 'Politikë': 5, 'Showbiz': 6, 'Shëndet': 7, 'Sport': 8}


,text_final,label,source,date,label_id
0,Gjëra tronditëse që mësuam nga Wikileaks: Fiks...,Botë,Bota_al,2016-10-14T16:17:57,0
1,A është argjentinasja ajo që i shton gjasat që...,Showbiz,GazetaBlic,2017-01-18 11:35:55,6
2,"Ronaldo, Aguerro apo Benzema? Cana tregon se c...",Sport,GazetaBlic,2017-01-23 11:50:40,8
3,"Luftojnë tensionin e lartë të gjakut, këto jan...",Shëndet,Lajme_al,2021-04-02 17:33:17,7
4,"Huamarrja me euro, qeveria ndryshon monedhën e...",Ekonomi,Lajme_al,2016-11-21 17:23:46,1


tokenizim me XLM-RoBERTa

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[['text_final', 'label_id']])
val_ds = Dataset.from_pandas(val_df[['text_final', 'label_id']])
test_ds = Dataset.from_pandas(test_df[['text_final', 'label_id']])

def tokenize_function(examples):
    return tokenizer(examples['text_final'], truncation=True, max_length=256, padding='max_length')

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.rename_column('label_id', 'labels')
val_ds = val_ds.rename_column('label_id', 'labels')
test_ds = test_ds.rename_column('label_id', 'labels')

train_ds = train_ds.remove_columns(['text_final'])
val_ds = val_ds.remove_columns(['text_final'])
test_ds = test_ds.remove_columns(['text_final'])

print(train_ds)
print("\nShembull i tokenizuar:")
print(train_ds[0])

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/75480 [00:00<?, ? examples/s]

Map:   0%|          | 0/16174 [00:00<?, ? examples/s]

Map:   0%|          | 0/16175 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 75480
})

Shembull i tokenizuar:
{'labels': 0, 'input_ids': [0, 527, 14381, 219, 52062, 7564, 76066, 815, 1319, 1159, 302, 817, 39418, 133, 24251, 12, 75837, 62318, 452, 164366, 9, 272, 5, 12481, 10570, 28, 5558, 134, 36365, 173877, 46, 467, 102592, 2175, 46, 521, 16021, 815, 130410, 322, 12015, 34, 6796, 4, 541, 461, 185927, 4, 322, 25463, 40191, 5, 2609, 3012, 85204, 27706, 156, 29701, 40, 4, 57, 25206, 815, 1534, 156, 119052, 322, 655, 236, 815, 54, 134, 1630, 3454, 2611, 1479, 115046, 3208, 109097, 4, 12094, 54, 134, 8, 26886, 19266, 21003, 134, 10570, 655, 4249, 28, 12015, 20129, 4353, 5, 5275, 16021, 25206, 4, 1193, 9482, 32614, 590, 7451, 106245, 867, 17, 79169, 18, 16042, 467, 104968, 67, 815, 14823, 2605, 17, 27450, 7451, 73828, 5, 5375, 75, 22430, 3116, 4, 1300, 184603, 118, 322, 1371, 16840, 40, 7304, 14823, 2605, 17, 23764, 26787, 20520, 7451, 104, 56350, 2605, 1319, 17, 16921, 17, 4477, 5, 120

ngarko modelin XLM-R dhe konfiguro metrics

In [6]:
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(kategorite),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro
    }

print("Modeli u ngarkua. Numri i klasave:", model.config.num_labels)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modeli u ngarkua. Numri i klasave: 9


konfiguro Training Arguments dhe Trainer

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='/kaggle/working/xlmr-results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=100,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Trainer u konfigurua. Gati për fine-tuning.")

Trainer u konfigurua. Gati për fine-tuning.


Fillo trajnimin

In [8]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.815877,0.793472,0.865958,0.864746
2,0.680853,0.734459,0.876345,0.875274
3,0.491041,0.719882,0.883022,0.881890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=7077, training_loss=0.767787183562426, metrics={'train_runtime': 7581.0993, 'train_samples_per_second': 29.869, 'train_steps_per_second': 0.934, 'total_flos': 2.979130595899392e+16, 'train_loss': 0.767787183562426, 'epoch': 3.0})

vlerësimi mbi test set

In [9]:
test_results = trainer.evaluate(eval_dataset=test_ds)
print("Rezultatet mbi TEST SET:")
for key, value in test_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Rezultatet mbi TEST SET:
eval_loss: 0.7679405212402344
eval_accuracy: 0.8765996908809892
eval_f1_macro: 0.8755335687748921
eval_runtime: 157.0317
eval_samples_per_second: 103.005
eval_steps_per_second: 1.611
epoch: 3.0


Ruaj modelin dhe rezultatet

In [10]:
import json

trainer.save_model('/kaggle/working/xlmr-final-model')
tokenizer.save_pretrained('/kaggle/working/xlmr-final-model')

xlmr_rezultate = {
    'model': 'xlm-roberta-base',
    'test_accuracy': test_results['eval_accuracy'],
    'test_f1_macro': test_results['eval_f1_macro'],
    'test_loss': test_results['eval_loss'],
    'training_time_sec': 7581.0993,
    'epochs': 3
}

with open('/kaggle/working/xlmr_rezultate.json', 'w') as f:
    json.dump(xlmr_rezultate, f, indent=2)

print("Modeli dhe rezultatet u ruajtën.")
print(xlmr_rezultate)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modeli dhe rezultatet u ruajtën.
{'model': 'xlm-roberta-base', 'test_accuracy': 0.8765996908809892, 'test_f1_macro': 0.8755335687748921, 'test_loss': 0.7679405212402344, 'training_time_sec': 7581.0993, 'epochs': 3}


In [11]:
import shutil
shutil.make_archive('/kaggle/working/xlmr_model_backup', 'zip', '/kaggle/working/xlmr-final-model')
print("U krijua zip-i me sukses!")

import os
size_mb = os.path.getsize('/kaggle/working/xlmr_model_backup.zip') / (1024*1024)
print(f"Madhësia: {size_mb:.1f} MB")

U krijua zip-i me sukses!
Madhësia: 832.6 MB


In [3]:
import numpy as np
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

predictions_output = trainer.predict(test_ds)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

y_pred_labels = [id2label[i] for i in y_pred]
y_true_labels = [id2label[i] for i in y_true]

print("Classification Report (XLM-RoBERTa, Test Set):")
print(classification_report(y_true_labels, y_pred_labels))

NameError: name 'trainer' is not defined